# ⚡ SHAKTI — External Lab 1-Click GPU Continual Learning & Crore-Scale Auto-Shard Demo
**Zero-Secret Split-Brain SDK (`shakti_sdk.py` v3.1)**

Run verified continual learning on your own GPU (`Qwen`, `Llama`, `Gemma`, `Mistral`, `OLMoE`, or custom PyTorch models) using your assigned **`SHAKTI-KEY-...`**.
- **0% Catastrophic Forgetting** (`100.0%` retention, `98.8%–99.6%` unseen same-domain locality)
- **0% Adversarial Poisoning** (`100%` unverified/contradictory claim rejection)
- **Crore-Scale Auto-Shard Streaming (`train_stream`)** (`O(shard_size)` constant RAM, `0.0%` cross-shard interference, `< 0.1 ms` SQLite B-tree shard lookup)
- **Live Key Control** (Instant activation/revocation via SHAKTI Master Server)

In [ ]:
# 1. Download or load the SHAKTI Client Connector (0% proprietary formula inside)
import pathlib, urllib.request
if not pathlib.Path('shakti_sdk.py').exists():
    urllib.request.urlretrieve('https://raw.githubusercontent.com/MEMGATE/SHAKTI-CLOUD-KEYS/main/shakti_sdk.py', 'shakti_sdk.py')
import shakti_sdk
print('SHAKTI SDK v3.2 loaded successfully!')

In [ ]:
# 2. Connect with your assigned SHAKTI Access Key
SHAKTI_KEY = "SHAKTI-KEY-1001-C7AE"  # <-- Replace with your assigned key
trainer = shakti_sdk.connect(api_key=SHAKTI_KEY)
print("Connection Status:", trainer.verify())

In [ ]:
# 3. Quick Smoke Test (Poison Filtering + Session Preparation + Subspace Gate Verification)
!python shakti_sdk.py --key {SHAKTI_KEY} --test

In [ ]:
# 4. Full GPU Continual-Learning Run on Your HuggingFace Model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # Or Qwen2.5-7B-Instruct, Llama-3, OLMoE, etc.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

report = trainer.train(
    model=model,
    tokenizer=tokenizer,
    new_data=[
        {"question": "Who holds the office of Chief Data Officer of NovaGrid in 2026?", "answer": "Aarav Mehta", "verified": True},
        {"question": "Who holds the office of Chief Data Officer of NovaGrid in 2026?", "answer": "Fake Poison Actor", "poison": True},
        {"question": "What is the primary port of Cluster-9?", "answer": "Port 9443", "verified": True},
    ],
    anchors=[
        {"question": "What is 15 * 15?", "answer": "225"},
        {"question": "What is the capital of France?", "answer": "Paris"},
    ],
    unchanged_neighbours=[
        {"question": "Who holds the office of Chief Executive Officer of NovaGrid?", "answer": "Elena Rostova"},
    ],
    epochs=2,
    lr=6e-5,
)

import json
print(json.dumps({k: v for k, v in report.items() if k != 'subspace_gate'}, indent=2))

In [ ]:
# 5. Ask Trained Model Directly (Exact + Rephrased + Unseen Locality)
for q in [
    "Who holds the office of Chief Data Officer of NovaGrid in 2026?",
    "Please tell me who is the Chief Data Officer of NovaGrid in 2026?",
    "Who holds the office of Chief Executive Officer of NovaGrid?",
    "What is 15 * 15?",
]:
    ans = trainer.ask(model, tokenizer, q)
    print(f"Q: {q}\nA: {ans}\n" + "-" * 60)

In [ ]:
# 6. Crore-Scale Streaming Mode (`train_stream` — Auto-Shard Almari Engine)
def large_fact_stream(n_facts=10000):
    for i in range(n_facts):
        yield {"question": f"Who is the Regional Director of Sector {i}?", "answer": f"Director_{i}", "verified": True}

stream_report = trainer.train_stream(
    data_stream=large_fact_stream(10000),
    shard_size=500,
    train_weights_for_all_shards=False,
)
print(json.dumps(stream_report, indent=2))